# CS515 HW3 Analysis

This notebook follows the compact HW2 style: it reads saved artifacts from `artifacts/` and produces the tables and figures needed for the report without rerunning training.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.manifold import TSNE

ARTIFACTS_DIR = Path("artifacts")
RUNS_DIR = ARTIFACTS_DIR / "runs"
ROBUSTNESS_DIR = ARTIFACTS_DIR / "robustness"
FEATURES_DIR = ARTIFACTS_DIR / "features"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
SUMMARY_PATH = ARTIFACTS_DIR / "summary.csv"
FLOPS_PATH = ARTIFACTS_DIR / "flops_summary.csv"

NUMERIC_COLUMNS = {
    "epoch", "loss", "accuracy", "lr", "label_smoothing", "temperature", "alpha",
    "best_epoch", "best_val_loss", "best_val_accuracy", "final_train_loss",
    "final_train_accuracy", "final_val_loss", "final_val_accuracy", "test_loss",
    "test_accuracy", "severity", "epsilon", "steps", "num_samples", "clean_accuracy",
    "adversarial_accuracy", "macs", "params", "flops"
}

def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, keep_default_na=False)
    for column in sorted(NUMERIC_COLUMNS.intersection(frame.columns)):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame

run_frames = []
for csv_path in sorted(RUNS_DIR.glob("*.csv")):
    frame = load_csv(csv_path)
    frame["source_file"] = csv_path.name
    run_frames.append(frame)

robustness_frames = []
for csv_path in sorted(ROBUSTNESS_DIR.glob("*.csv")):
    frame = load_csv(csv_path)
    frame["source_file"] = csv_path.name
    robustness_frames.append(frame)

runs_df = pd.concat(run_frames, ignore_index=True) if run_frames else pd.DataFrame()
robustness_df = pd.concat(robustness_frames, ignore_index=True) if robustness_frames else pd.DataFrame()
summary_df = load_csv(SUMMARY_PATH)
flops_df = load_csv(FLOPS_PATH)

print("runs:", len(run_frames), "robustness files:", len(robustness_frames), "feature files:", len(list(FEATURES_DIR.glob("*.npz"))))


In [ ]:
core_runs = [
    "teacher_clean_resnet",
    "teacher_augmix_resnet",
    "student_distill_from_clean_teacher",
    "student_distill_from_augmix_teacher",
]

if not summary_df.empty:
    cols = ["run_name", "model", "train_mode", "teacher_run", "best_val_accuracy", "test_accuracy"]
    display(summary_df[summary_df["run_name"].isin(core_runs)][cols].sort_values("run_name"))
else:
    print("No summary.csv found yet.")


In [ ]:
def plot_accuracy_curves(run_names, title):
    if runs_df.empty:
        print("No run CSVs found yet.")
        return
    subset = runs_df[runs_df["run_name"].isin(run_names) & runs_df["split"].isin(["train", "val"])]
    if subset.empty:
        print("No rows for", run_names)
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for split, axis in zip(["train", "val"], axes):
        split_df = subset[subset["split"] == split]
        for run_name in run_names:
            run_df = split_df[split_df["run_name"] == run_name]
            if not run_df.empty:
                axis.plot(run_df["epoch"], run_df["accuracy"], marker="o", label=run_name)
        axis.set_title(f"{title} ({split})")
        axis.set_xlabel("Epoch")
        axis.set_ylabel("Accuracy")
        axis.grid(alpha=0.3)
        axis.legend()
    plt.tight_layout()

plot_accuracy_curves(["teacher_clean_resnet", "teacher_augmix_resnet"], "Teacher Comparison")
plot_accuracy_curves(["student_distill_from_clean_teacher", "student_distill_from_augmix_teacher"], "Student Comparison")


In [ ]:
if robustness_df.empty:
    print("No robustness CSVs found yet.")
else:
    cifar10c_df = robustness_df[robustness_df["eval_type"] == "cifar10c"].copy()
    attack_df = robustness_df[robustness_df["eval_type"].isin(["pgd", "transfer"])].copy()
    
    if not cifar10c_df.empty:
        mean_corruption = (
            cifar10c_df[cifar10c_df["corruption_name"] != "clean"]
            .groupby("run_name", as_index=False)["clean_accuracy"]
            .mean()
            .rename(columns={"clean_accuracy": "mean_cifar10c_accuracy"})
        )
        clean_rows = cifar10c_df[cifar10c_df["corruption_name"] == "clean"][["run_name", "clean_accuracy"]]
        merged = clean_rows.merge(mean_corruption, on="run_name", how="left")
        display(merged.sort_values("run_name"))

        plot_df = merged[merged["run_name"].isin(core_runs)]
        if not plot_df.empty:
            x = np.arange(len(plot_df))
            width = 0.35
            plt.figure(figsize=(10, 4))
            plt.bar(x - width / 2, plot_df["clean_accuracy"], width=width, label="Clean")
            plt.bar(x + width / 2, plot_df["mean_cifar10c_accuracy"], width=width, label="Mean CIFAR-10-C")
            plt.xticks(x, plot_df["run_name"], rotation=25, ha="right")
            plt.ylabel("Accuracy")
            plt.title("Clean vs Corruption Accuracy")
            plt.legend()
            plt.tight_layout()

    if not attack_df.empty:
        display(attack_df[["run_name", "eval_type", "attack_norm", "clean_accuracy", "adversarial_accuracy"]].sort_values(["eval_type", "run_name", "attack_norm"]))
        plot_df = attack_df[attack_df["eval_type"] == "pgd"]
        if not plot_df.empty:
            pivot = plot_df.pivot_table(index="run_name", columns="attack_norm", values="adversarial_accuracy", aggfunc="mean")
            pivot.loc[[name for name in core_runs if name in pivot.index]].plot(kind="bar", figsize=(10, 4))
            plt.ylabel("Adversarial Accuracy")
            plt.title("PGD Robustness")
            plt.tight_layout()


In [ ]:
if robustness_df.empty:
    print("No robustness CSVs found yet.")
else:
    teacher_corruptions = robustness_df[
        (robustness_df["eval_type"] == "cifar10c") &
        (robustness_df["corruption_name"] != "clean") &
        (robustness_df["run_name"].isin(["teacher_clean_resnet", "teacher_augmix_resnet"]))
    ].copy()
    if not teacher_corruptions.empty:
        grouped = (
            teacher_corruptions.groupby(["corruption_name", "run_name"], as_index=False)["clean_accuracy"].mean()
        )
        pivot = grouped.pivot(index="corruption_name", columns="run_name", values="clean_accuracy").sort_index()
        pivot.plot(kind="bar", figsize=(12, 4))
        plt.ylabel("Mean Accuracy")
        plt.title("Teacher Corruption Breakdown")
        plt.tight_layout()
    else:
        print("No teacher CIFAR-10-C rows found yet.")


In [ ]:
feature_files = sorted(FEATURES_DIR.glob("*.npz"))
if not feature_files:
    print("No feature exports found yet.")
else:
    feature_path = feature_files[0]
    data = np.load(feature_path, allow_pickle=True)
    features = data["features"]
    labels = data["labels"]
    kinds = data["sample_kind"]
    
    max_points = min(len(features), 2000)
    indices = np.arange(len(features))[:max_points]
    embedding = TSNE(n_components=2, random_state=42, init="pca", learning_rate="auto").fit_transform(features[indices])
    frame = pd.DataFrame({
        "x": embedding[:, 0],
        "y": embedding[:, 1],
        "label": labels[indices],
        "sample_kind": kinds[indices],
    })
    plt.figure(figsize=(7, 6))
    for sample_kind, subset in frame.groupby("sample_kind"):
        plt.scatter(subset["x"], subset["y"], s=10, alpha=0.5, label=sample_kind)
    plt.title(f"t-SNE from {feature_path.name}")
    plt.legend()
    plt.tight_layout()


In [ ]:
figure_files = sorted(FIGURES_DIR.glob("*.png"))
if not figure_files:
    print("No Grad-CAM figures found yet.")
else:
    for figure_path in figure_files[:4]:
        print(figure_path.name)
        display(Image(filename=str(figure_path)))


In [ ]:
if not flops_df.empty:
    display(flops_df)
    flops_df.plot(x="arch_name", y=["params", "flops"], kind="bar", figsize=(8, 4), subplots=True, layout=(1, 2), legend=False)
    plt.tight_layout()
else:
    print("No FLOPs summary found yet.")
